In [1]:
import numpy as np

data = np.load("dataset_mc_ml_v1_prepared.npz")

X_train = data["X_train"]
X_val   = data["X_val"]
X_test  = data["X_test"]

y_train = data["y_train"]
y_val   = data["y_val"]
y_test  = data["y_test"]


In [2]:
def build_regime_labels(y):
    # 1 = cost-driven, 0 = congestion-driven
    return (y[:, 0] > y[:, 1]).astype(int)

r_train = build_regime_labels(y_train)
r_val   = build_regime_labels(y_val)
r_test  = build_regime_labels(y_test)


In [4]:
def balance_dataset(X, r, seed=0):
    np.random.seed(seed)
    idx0 = np.where(r == 0)[0]
    idx1 = np.where(r == 1)[0]

    n = min(len(idx0), len(idx1))

    idx0_sel = np.random.choice(idx0, n, replace=False)
    idx1_sel = np.random.choice(idx1, n, replace=False)

    idx = np.concatenate([idx0_sel, idx1_sel])
    np.random.shuffle(idx)

    return X[idx], r[idx]


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix

X_train_bal, r_train_bal = balance_dataset(X_train, r_train)

logit = LogisticRegression(max_iter=1000)
logit.fit(X_train_bal, r_train_bal)

r_test_pred = logit.predict(X_test)
r_test_prob = logit.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(r_test, r_test_pred))
print("ROC-AUC:", roc_auc_score(r_test, r_test_prob))
print(confusion_matrix(r_test, r_test_pred))

print("=== Logistic regression ===")
print(f"Accuracy: {acc:.3f}")
print(f"ROC-AUC:  {auc:.3f}")
print("Confusion matrix:")
print(confusion_matrix(r_test, r_test_pred))


Accuracy: 0.4666666666666667
ROC-AUC: 0.6666666666666667
[[ 2  1]
 [15 12]]
=== Logistic regression ===
Accuracy: 0.900
ROC-AUC:  0.420
Confusion matrix:
[[ 2  1]
 [15 12]]
